# Gold 01 — Daily store sales

**Grain:** one row for each calendar date and store, including zero-sale days. The date range comes from Silver orders and the store list comes from Silver stores.

**Sales policy:** only completed orders count. Revenue is subtotal minus discount, before tax. Tax and the final amount including tax stay in separate columns. Estimated gross profit is net revenue minus estimated item cost. Average order value is net revenue divided by completed orders. These are explicit project metric definitions, because the assignment does not define revenue's tax treatment.

Run after Silver order_items includes estimated_line_cost and estimated_line_margin. The notebook first reduces items to one row per order, then joins them to order headers. This avoids multiplying the order amount by the number of items.


In [ ]:
import os  # Set Java before Spark starts.
import shutil  # Find Linux Java.
from pathlib import Path  # Build data paths.
from pyspark.sql import SparkSession, functions as F  # Use Spark and column expressions.
from delta import configure_spark_with_delta_pip  # Add Delta Lake to Spark.
java = shutil.which("java")  # Locate WSL Java.
if java is None:  # Require Java.
    raise RuntimeError("Java is unavailable in WSL")  # Explain a startup failure.
os.environ["JAVA_HOME"] = str(Path(java).resolve().parent.parent)  # Avoid the Windows Java path.
project_root = Path("/mnt/f/Big Data/Retail_Lakehouse_Learning")  # Find the project in WSL.
if not project_root.is_dir():  # Check the mounted drive.
    raise FileNotFoundError(project_root)  # Stop on an incorrect path.
builder = (SparkSession.builder.master("local[2]")  # Use two local threads.
    .appName("RetailGoldDailyStoreSales")  # Label this job.
    .config("spark.sql.shuffle.partitions", "4")  # Keep small local shuffles manageable.
    .config("spark.ui.enabled", "false")  # Avoid a second Spark UI port.
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")  # Enable Delta SQL.
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog"))  # Register Delta.
spark = configure_spark_with_delta_pip(builder).getOrCreate()  # Start or reuse Spark.
print("Spark:", spark.version, "Timezone:", spark.conf.get("spark.sql.session.timeZone"))  # Record session behavior.


In [ ]:
silver_root = project_root / "lakehouse" / "silver" / "data"  # Locate cleaned source tables.
orders_df = spark.read.format("delta").load(str(silver_root / "orders"))  # Read typed orders.
items_df = spark.read.format("delta").load(str(silver_root / "order_items"))  # Read typed items and costs.
stores_df = spark.read.format("delta").load(str(silver_root / "stores"))  # Read cleaned stores.
needed_item_columns = {"order_id", "quantity", "line_total", "estimated_line_cost", "estimated_line_margin"}  # Define required line fields.
if not needed_item_columns.issubset(items_df.columns):  # Require the updated Silver schema.
    raise ValueError("Rerun Silver order_items to add estimated cost and margin")  # Prevent incomplete profit.
if stores_df.select("store_id").distinct().count() != stores_df.count():  # Require one row per store.
    raise ValueError("Duplicate Silver store keys")  # Prevent a multiplied calendar.
print("Silver orders:", orders_df.count(), "items:", items_df.count(), "stores:", stores_df.count())  # Show inputs.


In [ ]:
completed_orders_df = (orders_df  # Start with one row per order.
    .filter(F.col("status") == "completed")  # Exclude cancelled and returned orders from sales.
    .withColumn("date", F.to_date("order_ts"))  # Get the calendar date for daily grouping.
    .withColumn("revenue", (F.col("subtotal") - F.col("discount_amount")).cast("decimal(18,2)")))  # Net sales before tax.
completed_count = completed_orders_df.count()  # Count sales orders.
if completed_count == 0:  # A baseline with no completed orders needs investigation.
    raise ValueError("No completed orders available for Gold")  # Stop before saving empty KPIs.
invalid_header_count = completed_orders_df.filter(  # Check fields Gold will calculate with.
    F.col("date").isNull() | F.col("store_id").isNull()  # Require the day and store.
    | F.col("revenue").isNull() | (F.col("revenue") < 0)  # Require nonnegative net revenue.
).count()  # Count invalid sales headers.
if invalid_header_count:  # Apply the Gold input gate.
    raise ValueError(f"{invalid_header_count} invalid completed orders")  # Keep the prior Gold version.
print("Completed sales orders:", completed_count)  # Show the business filter result.


In [ ]:
items_by_order_df = items_df.groupBy("order_id").agg(  # Reduce many lines to one row per order.
    F.sum("quantity").alias("units_sold"),  # Count all units in the order.
    F.sum("line_total").cast("decimal(18,2)").alias("item_subtotal"),  # Reconstruct pre-discount subtotal.
    F.sum("estimated_line_cost").cast("decimal(18,2)").alias("estimated_cost"),  # Sum product-based line costs.
)  # Keep the order_id key.
orders_with_items_df = completed_orders_df.join(items_by_order_df, on="order_id", how="left")  # Keep each completed order.
if orders_with_items_df.count() != completed_count:  # One-to-many item joins must not multiply headers.
    raise ValueError("Item aggregation changed the completed-order grain")  # Stop double counting.
bad_item_count = orders_with_items_df.filter(  # Find missing items or inconsistent subtotals.
    F.col("units_sold").isNull() | (F.col("units_sold") <= 0)  # Every completed order needs positive units.
    | F.col("estimated_cost").isNull()  # Every order needs estimated cost.
    | (F.abs(F.col("subtotal") - F.col("item_subtotal")) > F.lit(0.02))  # Compare source line sum with header.
).count()  # Count failures.
if bad_item_count:  # Enforce the financial grain check.
    raise ValueError(f"{bad_item_count} completed orders lack consistent items")  # Preserve existing Gold.
print("Completed orders with item totals:", orders_with_items_df.count())  # Confirm the safe join.


In [ ]:
daily_facts_df = orders_with_items_df.groupBy("date", "store_id").agg(  # Aggregate order-grain rows by day and store.
    F.count("*").alias("orders"),  # Count completed orders.
    F.sum("units_sold").alias("units_sold"),  # Add item units across orders.
    F.sum("revenue").cast("decimal(18,2)").alias("revenue"),  # Net revenue excluding tax.
    F.sum("estimated_cost").cast("decimal(18,2)").alias("estimated_cost"),  # Estimated product costs.
    F.sum("discount_amount").cast("decimal(18,2)").alias("discount_amount"),  # Discounts granted.
    F.sum("tax_amount").cast("decimal(18,2)").alias("tax_amount"),  # Tax collected.
    F.sum("total_amount").cast("decimal(18,2)").alias("amount_including_tax"),  # Final customer amount.
)
date_bounds = orders_df.select(  # Find the full observed order date range.
    F.min(F.to_date("order_ts")).alias("first_date"),  # Earliest order date.
    F.max(F.to_date("order_ts")).alias("last_date"),  # Latest order date.
).first()  # Bring two dates to the Python driver.
if date_bounds.first_date is None or date_bounds.last_date is None:  # Require a nonempty date range.
    raise ValueError("Order dates are missing")  # Stop before making the calendar.
dates_df = spark.range(1).select(  # Create one row as a starting point for a date sequence.
    F.explode(F.sequence(F.lit(date_bounds.first_date), F.lit(date_bounds.last_date))).alias("date")  # One row per day.
)
store_dims_df = stores_df.select("store_id", "store_name", "city", "region")  # Keep display fields.
store_calendar_df = dates_df.crossJoin(store_dims_df)  # Give every store every date, including zero-sales dates.
gold_daily_df = store_calendar_df.join(daily_facts_df, on=["date", "store_id"], how="left")  # Preserve the full calendar.
zero_money = F.lit("0.00").cast("decimal(18,2)")  # Reuse an exact zero money value.
for name in ("revenue", "estimated_cost", "discount_amount", "tax_amount", "amount_including_tax"):  # Fill missing sales facts.
    gold_daily_df = gold_daily_df.withColumn(name, F.coalesce(F.col(name), zero_money))  # Keep no-sale days as zero.
for name in ("orders", "units_sold"):  # Fill the two whole-number measures.
    gold_daily_df = gold_daily_df.withColumn(name, F.coalesce(F.col(name), F.lit(0)))  # Use zero counts.
gold_daily_df = gold_daily_df.withColumn(  # Compute estimated profit on net sales.
    "estimated_gross_profit", (F.col("revenue") - F.col("estimated_cost")).cast("decimal(18,2)")  # Exclude tax.
).withColumn(  # Define average completed order value.
    "avg_order_value", F.when(F.col("orders") > 0,  # Divide only on sales days.
        F.bround(F.col("revenue") / F.col("orders"), 2).cast("decimal(18,2)")  # Round to cents.
    ).otherwise(zero_money)  # Zero for store-days with no completed order.
)
gold_daily_df = gold_daily_df.select(  # Choose stable, readable Gold columns.
    "date", "store_id", "store_name", "city", "region", "orders", "units_sold",
    "revenue", "estimated_cost", "estimated_gross_profit", "avg_order_value",
    "discount_amount", "tax_amount", "amount_including_tax",
)
print("Sales store-days:", daily_facts_df.count(), "Calendar store-days:", gold_daily_df.count())  # See zero days.
gold_daily_df.orderBy("date", "store_id").show(5, truncate=False)  # Preview the mart.


In [ ]:
expected_rows = dates_df.count() * stores_df.count()  # Count date-store combinations.
if gold_daily_df.count() != expected_rows:  # Enforce the full calendar grain.
    raise ValueError("Gold calendar has missing or multiplied rows")  # Stop on a row-count mismatch.
duplicate_keys = gold_daily_df.groupBy("date", "store_id").count().filter(F.col("count") > 1).count()  # DQ009.
if duplicate_keys:  # A Gold day/store key must be unique.
    raise ValueError(f"DQ009: {duplicate_keys} duplicate date/store keys")  # Fail Gold build.
order_revenue = completed_orders_df.agg(F.sum("revenue").alias("value")).first().value  # Expected net revenue.
gold_revenue = gold_daily_df.agg(F.sum("revenue").alias("value")).first().value  # Stored-day revenue before write.
if order_revenue != gold_revenue:  # Check the full calendar did not alter money.
    raise ValueError(f"Gold revenue mismatch: {gold_revenue} versus {order_revenue}")  # Stop on a broken aggregation.
order_units = orders_with_items_df.agg(F.sum("units_sold").alias("value")).first().value  # Expected units.
gold_units = gold_daily_df.agg(F.sum("units_sold").alias("value")).first().value  # Daily units.
if order_units != gold_units:  # Reconcile the item measure.
    raise ValueError("Gold units do not match completed order items")  # Stop on missing units.
print("DQ009 duplicate keys:", duplicate_keys, "Revenue:", gold_revenue, "Units:", gold_units)  # Show gates.
print("Zero-sale store-days:", gold_daily_df.filter(F.col("orders") == 0).count())  # Preserve no-sale observations.


In [ ]:
gold_path = project_root / "lakehouse" / "gold" / "data" / "daily_store_sales"  # Choose the Gold Delta folder.
gold_path.parent.mkdir(parents=True, exist_ok=True)  # Ensure the ignored data parent exists.
gold_daily_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(str(gold_path))  # Save one snapshot.
saved_gold_df = spark.read.format("delta").load(str(gold_path))  # Read the committed version.
if saved_gold_df.count() != expected_rows:  # Verify the persisted grain count.
    raise ValueError("Saved Gold daily store row count is wrong")  # Report a failed reconciliation.
if saved_gold_df.agg(F.sum("revenue").alias("value")).first().value != order_revenue:  # Verify persisted revenue.
    raise ValueError("Saved Gold daily store revenue is wrong")  # Report incorrect money.
print("Saved daily_store_sales:", saved_gold_df.count())  # Show the Gold row count.
print("Delta log exists:", (gold_path / "_delta_log").is_dir())  # Confirm a committed Delta table.
saved_gold_df.printSchema()  # Show Power BI-ready types.
